In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import date

PARK_ID = 160

def scrape_day(d):
    url = f"https://queue-times.com/parks/{PARK_ID}/calendar/{d:%Y/%m/%d}"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})

    if response.status_code != 200:
        print(f"Kon pagina niet ophalen voor {d}, status {response.status_code}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # 1. Crowd level
    crowd_percent = None
    crowd_label = None
    for block in soup.select("div.panel-block"):
        spans = block.find_all("span")
        if spans and spans[0].get_text(strip=True) == "Crowd level":
            val = spans[1].get_text(strip=True)
            if "%" in val:
                crowd_percent = val
            else:
                crowd_label = val

    # 2. Temperature
    temperature_forecast = None
    temperature_actual = None
    for h2 in soup.select("div.panel .panel-heading h2"):
        if "temperature" in h2.get_text(strip=True).lower():
            panel = h2.find_parent("div", class_="panel")
            for block in panel.select("div.panel-block"):
                spans = block.find_all("span")
                if len(spans) >= 2:
                    key = spans[0].get_text(strip=True).lower()
                    val = spans[1].get_text(strip=True)
                    if "forecast average" in key:
                        temperature_forecast = val
                    elif "actual average" in key:
                        temperature_actual = val

    # 3. Precipitation
    intensity_forecast = None
    intensity_actual = None
    for h2 in soup.select("div.panel .panel-heading h2"):
        if "precipitation" in h2.get_text(strip=True).lower():
            panel = h2.find_parent("div", class_="panel")
            for block in panel.select("div.panel-block"):
                spans = block.find_all("span")
                if len(spans) >= 2:
                    key = spans[0].get_text(strip=True).lower()
                    val = spans[1].get_text(strip=True)
                    if "forecast average" in key:
                        intensity_forecast = val
                    elif "actual average" in key:
                        intensity_actual = val

    # 4. Wind
    wind_forecast = None
    wind_actual = None
    for h2 in soup.select("div.panel .panel-heading h2"):
        if "wind speed" in h2.get_text(strip=True).lower():
            panel = h2.find_parent("div", class_="panel")
            for block in panel.select("div.panel-block"):
                spans = block.find_all("span")
                if len(spans) >= 2:
                    key = spans[0].get_text(strip=True).lower()
                    val = spans[1].get_text(strip=True)
                    if "forecast average" in key:
                        wind_forecast = val
                    elif "actual average" in key:
                        wind_actual = val

    # 5. Events
    events = None
    for h2 in soup.select("div.panel .panel-heading h2"):
        if "events" in h2.get_text(strip=True).lower():
            panel = h2.find_parent("div", class_="panel")
            names = []
            for block in panel.select("div.panel-block"):
                span = block.find("span")
                if span:
                    txt = span.get_text(strip=True)
                    if txt:
                        names.append(txt)
            events = "; ".join(names) if names else None
            break

    return {
        "date": d.isoformat(),
        "crowd_percent": crowd_percent,
        "crowd_label": crowd_label,
        "temperature_forecast": temperature_forecast,
        "temperature_actual": temperature_actual,
        "intensity_forecast": intensity_forecast,
        "intensity_actual": intensity_actual,
        "wind_forecast": wind_forecast,
        "wind_actual": wind_actual,
        "events": events,
    }

if __name__ == "__main__":
    scrape_date = date(2023, 1, 1)
    data = scrape_day(scrape_date)
    if data:
        for key, value in data.items():
            print(f"{key}: {value}")